### Deep agent with real persistent stores and DB:

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")
os.environ["PG_DATABASE_URL"]=os.getenv("PG_DATABASE_URL")

#### Initialize model for the use:

In [3]:
from langchain_ollama import ChatOllama

model = ChatOllama(
    model="gemma4:e4b",
    temperature=0.3
)

response = model.invoke("Explain postgres SQL database and mongoDB DB.")

print(response.content)

This is a fantastic question because PostgreSQL and MongoDB represent the two major philosophical approaches to data storage today: **Structure vs. Flexibility.**

While both are powerful tools for storing and retrieving information, they operate on fundamentally different principles, which dictates what kind of applications they are best suited for.

Here is a detailed explanation of both databases, followed by a direct comparison.

---

## 🐘 1. PostgreSQL (The Structured Approach)

PostgreSQL is a powerful, open-source **Relational Database Management System (RDBMS)**. It is the gold standard for structured data and data integrity.

### 💡 How It Works: The Relational Model
PostgreSQL organizes data into **tables**. Think of a table like a highly organized spreadsheet.

*   **Tables:** Each table represents a specific entity (e.g., `Users`, `Products`, `Orders`).
*   **Rows:** Each row is a single record (e.g., one specific user).
*   **Columns:** Each column defines a specific attrib

In [2]:
from langchain_ollama import ChatOllama

model = ChatOllama(
    model="gemma4:e4b",
    temperature=0.3
)

### Configuring Postgres SQL DB for persistent store: 

### implementing User + Thread isolation (namespace)

In [19]:
import os
from dotenv import load_dotenv
from deepagents import create_deep_agent
from deepagents.backends import StoreBackend
from langgraph.store.postgres import PostgresStore

load_dotenv()
postgres_url = os.getenv("PG_DATABASE_URL")

with PostgresStore.from_conn_string(postgres_url) as postgres_store:
    postgres_store.setup()  
    
    agent = create_deep_agent(
        model=model,
        backend=StoreBackend(
            namespace=lambda rt: (
                # 1. In production, use the secure server-injected user identity
                # 2. Locally, check if a custom user_id was manually passed in context
                # 3. Otherwise, fall back to a default notebook string
                rt.server_info.user.identity if rt.server_info 
                else getattr(rt.context, "user_id", "default_local_user"),
                
                rt.execution_info.thread_id, 
                "filesystem"
            ), 
        ),
        store=postgres_store
    )
    
    # --- SIMULATING USER A ---
    print("Executing as User A...")
    config_user_a = {
        "configurable": {"thread_id": "chat_session_100"},
        "context": {"user_id": "user_alex"} # Manually passing user context locally
    }
    agent.invoke({"messages": [{"role": "user", "content": "My name is Alex."}]}, config=config_user_a)
    
    # --- SIMULATING USER B ---
    print("Executing as User B...")
    config_user_b = {
        "configurable": {"thread_id": "chat_session_200"},
        "context": {"user_id": "user_blake"} # A completely different user
    }
    agent.invoke({"messages": [{"role": "user", "content": "My name is Blake."}]}, config=config_user_b)

Executing as User A...
Executing as User B...


In [20]:
# Run this in your separate cell
with PostgresStore.from_conn_string(postgres_url) as temp_store:
    results = temp_store.search(
        ("user_alex", "chat_session_100", "filesystem"), 
        limit=5
    )
    
    if not results:
        print("Empty store! No data found matching this namespace.")
    for item in results:
        print(f"Key: {item.key} | Value: {item.value}")

Empty store! No data found matching this namespace.
